# PulseVector Classification Analysis
This notebook provides a concise, reproducible entry point for inspecting the approved dataset and generated metrics. The production pipeline lives in `ml/` and `scripts/`.

**Disclaimer:** This project is an educational machine learning demonstration and is not a medical diagnostic tool.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET = PROJECT_ROOT / 'data' / 'raw' / 'heart_disease_cleveland.csv'
REPORT = PROJECT_ROOT / 'reports' / 'metrics.json'


In [ ]:
data = pd.read_csv(DATASET)
data.head()


In [ ]:
data.info()
data.isna().sum()[lambda s: s > 0]


In [ ]:
binary_target = (data['num'] > 0).astype(int)
binary_target.value_counts().sort_index()


In [ ]:
metrics = json.loads(REPORT.read_text())
comparison = pd.DataFrame({
    payload['display_name']: {
        'CV F1': payload['cross_validation']['f1']['mean'],
        'CV Recall': payload['cross_validation']['recall']['mean'],
        'Test F1': payload['test_metrics']['f1'],
    }
    for payload in metrics['candidate_models'].values()
}).T
comparison.sort_values('CV F1', ascending=False)


## Reproduce the full system
Run `python scripts/run_pipeline.py` from the project root. The script trains all five candidates, selects the cross-validation winner, evaluates the held-out test set, and regenerates models, reports, predictions, and charts.